# DocTR — Document Text Recognition Pipeline

In [ ]:
!pip install -q "python-doctr[torch]" pillow numpy

In [ ]:
import reimport numpy as npfrom PIL import Image, ImageDraw, ImageFontfrom doctr.io import DocumentFilefrom doctr.models import ocr_predictor

In [ ]:
DATASET_PATH = ""GROUND_TRUTH_TEXT = "DocTR combines a detection model with a recognition model for full page OCR."DET_ARCH = "db_resnet50"RECO_ARCH = "crnn_vgg16_bn"

In [ ]:
def create_sample_image(text, width=900, height=220, font_size=28):    image = Image.new("RGB", (width, height), color=(255, 255, 255))    draw = ImageDraw.Draw(image)    try:        font = ImageFont.truetype("DejaVuSans.ttf", font_size)    except IOError:        font = ImageFont.load_default()    draw.text((20, height // 2 - font_size), text, fill=(0, 0, 0), font=font)    return imagedef load_document(dataset_path, fallback_text):    if dataset_path:        if dataset_path.lower().endswith(".pdf"):            return DocumentFile.from_pdf(dataset_path)        return DocumentFile.from_images(dataset_path)    sample_image = create_sample_image(fallback_text)    sample_image.save("/tmp/sample_doctr_image.png")    return DocumentFile.from_images("/tmp/sample_doctr_image.png")doc = load_document(DATASET_PATH, GROUND_TRUTH_TEXT)

In [ ]:
model = ocr_predictor(det_arch=DET_ARCH, reco_arch=RECO_ARCH, pretrained=True)

In [ ]:
result = model(doc)

In [ ]:
exported = result.export()for page in exported["pages"]:    for block in page["blocks"]:        for line in block["lines"]:            for word in line["words"]:                print(word["value"], word["confidence"], word["geometry"])

In [ ]:
def draw_doctr_boxes(page_image, page_export):    boxed = page_image.convert("RGB").copy()    draw = ImageDraw.Draw(boxed)    width, height = boxed.size    for block in page_export["blocks"]:        for line in block["lines"]:            for word in line["words"]:                (x0, y0), (x1, y1) = word["geometry"]                box = [x0 * width, y0 * height, x1 * width, y1 * height]                draw.rectangle(box, outline=(255, 0, 0), width=2)    return boxedpage_image = Image.fromarray((doc[0] * 255).astype(np.uint8)) if isinstance(doc[0], np.ndarray) else Image.fromarray(doc[0])boxed_image = draw_doctr_boxes(page_image, exported["pages"][0])boxed_image

In [ ]:
def merge_recognized_text(page_export):    words = []    for block in page_export["blocks"]:        for line in block["lines"]:            for word in line["words"]:                words.append(word["value"])    return " ".join(words)recognized_text = merge_recognized_text(exported["pages"][0])print(recognized_text)

In [ ]:
def normalize_text(text, case_sensitive=False):    if not case_sensitive:        text = text.lower()    text = re.sub(r"\s+", " ", text).strip()    return list(text)def edit_distance_ops(ref_chars, hyp_chars):    n, m = len(ref_chars), len(hyp_chars)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_chars[i - 1] == hyp_chars[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i][j - 1], dp[i - 1][j])    return dp[n][m]def cer(reference, hypothesis):    ref_chars = normalize_text(reference)    hyp_chars = normalize_text(hypothesis)    distance = edit_distance_ops(ref_chars, hyp_chars)    n = len(ref_chars) if len(ref_chars) > 0 else 1    return distance / nprint("cer:", cer(GROUND_TRUTH_TEXT, recognized_text))